In [10]:
import sqlite3
from datetime import datetime

DB = 'lab3_blockchain.db'

def conn():
    c = sqlite3.connect(DB, timeout=5)
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

In [14]:
class Block:
    def __init__(self, id, view, desc):
        self.id = id
        self.view = view
        self.desc = desc

    @staticmethod
    def insert(id, view, desc):
        with conn() as c:
            c.execute("INSERT OR IGNORE INTO BLOCKS (id, view, desc) VALUES (?,?,?)", (id, view, desc))
            c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('block', id))

class Vote:
    def __init__(self, block_id, voter_id, timestamp, source_id):
        self.block_id = block_id
        self.voter_id = voter_id
        self.timestamp = timestamp
        self.source_id = source_id

    @staticmethod
    def insert(block_id, voter_id, timestamp, source_id):
        with conn() as c:
            c.execute("INSERT OR IGNORE INTO VOTES (block_id, voter_id, timestamp, source_id) VALUES (?,?,?,?)",
                      (block_id, voter_id, timestamp, source_id))
            c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)",
                      ('vote', f"{block_id}:{voter_id}"))

class Person:
    def __init__(self, id, name, addr):
        self.id = id
        self.name = name
        self.addr = addr

    @staticmethod
    def insert(id, name, addr):
        with conn() as c:
            c.execute("INSERT OR IGNORE INTO PERSONS (id, name, addr) VALUES (?,?,?)", (id, name, addr))
            c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('person', str(id)))

class Source:
    def __init__(self, id, ip, country):
        self.id = id
        self.ip = ip
        self.country = country

    @staticmethod
    def insert(id, ip, country):
        with conn() as c:
            c.execute("INSERT OR IGNORE INTO SOURCES (id, ip_addr, country_code) VALUES (?,?,?)", (id, ip, country))
            c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('source', str(id)))

In [18]:
class BlockProcessor:
    def __init__(self, poll_interval=5):
        self.poll_interval = poll_interval

    def process_one_event(self, event):
        typ = event['type']
        eid = event['entity_id']
        if typ == 'block':
            with conn() as c:
                row = c.execute("SELECT * FROM BLOCKS WHERE id=?", (eid,)).fetchone()
                print(f"Обробка блоку {row['id']}: {row['desc']}")
        elif typ == 'vote':
            bid, vid = eid.split(':')
            with conn() as c:
                row = c.execute("SELECT * FROM VOTES WHERE block_id=? AND voter_id=?", (bid, vid)).fetchone()
                print(f"Голос за блок {row['block_id']} від {row['voter_id']}")
        elif typ == 'person':
            with conn() as c:
                row = c.execute("SELECT * FROM PERSONS WHERE id=?", (eid,)).fetchone()
                print(f"Особа {row['id']}: {row['name']}")
        elif typ == 'source':
            with conn() as c:
                row = c.execute("SELECT * FROM SOURCES WHERE id=?", (eid,)).fetchone()
                print(f"Джерело {row['id']}: {row['ip_addr']}")

    def process_all_pending(self):
        with conn() as c:
            events = c.execute("SELECT id, type, entity_id FROM event_stream WHERE processed=0").fetchall()
        for ev in events:
            self.process_one_event(ev)
            with conn() as c:
                c.execute("UPDATE event_stream SET processed=1 WHERE id=?", (ev['id'],))

    def run(self):
        print("BlockProcessor запущено. Перевірка кожні", self.poll_interval, "сек.")
        while True:
            self.process_all_pending()
            time.sleep(self.poll_interval)

In [19]:
Block.insert('0xabc', 100, 'Перший блок')
Vote.insert('0xabc', 1, '2025-03-23 12:00', 2)
Person.insert(1, 'Владислав', 'Дніпро')
Source.insert(2, '192.168.1.1', 'UA')

bp = BlockProcessor()
bp.process_all_pending()

Обробка блоку 0xabc: Перший блок
Голос за блок 0xabc від 1
Особа 1: Катерина Слободяник
Джерело 2: 198.51.100.2
